In [13]:
import torch
import cv2
import numpy as np
from depth_anything_3.api import DepthAnything3

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


## materials used:

- Depth Anything 3 docs - https://github.com/ByteDance-Seed/Depth-Anything-3/blob/main/docs/API.md

- gsplat docs - https://docs.gsplat.studio/main/

- 3D Gaussian Splatting | 3DGS Implementation from Scratch in PyTorch-Only - https://www.youtube.com/watch?v=ZXmN2gWPHus

- https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/


- From Images to Semantic 3D Gaussian Splatting with Python - https://medium.com/data-science-collective/from-images-to-semantic-3d-gaussian-splatting-with-python-complete-guide-ff9d3d240847

In [23]:
def load_model(model="depth-anything/da3-base"):
    # 1. Setup GPU device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2. Load the model directly from Hugging Face
    # Options: "depth-anything/da3-small", "depth-anything/da3-base", "depth-anything/da3-large"
    model = DepthAnything3.from_pretrained("depth-anything/da3-base")
    model = model.to(device)

    return model, device

model, device = load_model()

print('model loaded')

[INFO ] using MLP layer as FFN
model loaded


In [24]:
import os

def get_images_paths(image_folder):
    image_paths = []

    for n, entry in enumerate(os.scandir(image_folder)):  
        if entry.is_file():
            image_paths.append(entry.path)
    print(f"from folder - {image_folder} - {n} images")

    return image_paths

image_folder = '/home/maciej/dev/depth/car_img'
image_paths = get_images_paths(image_folder)

from folder - /home/maciej/dev/depth/car_img - 9 images


In [41]:
def run_model_inference(model, image_paths):
    # 4. Run inference (Returns a dictionary with depth and metadata)
    prediction = model.inference(
        image=image_paths,
    )

    # Extract raw depth data for your point cloud pipeline
    # Note: V3 outputs raw depth and confidence metadata
    depth_map = prediction.depth
    confidence_map = prediction.conf # Useful for filtering out noise

    return depth_map, confidence_map


depth_map, confidence_map = run_model_inference(model, image_paths)

[WARN ] Images in batch have different sizes [(336, 504), (336, 504), (322, 504), (322, 504), (350, 504), (350, 504), (308, 504), (364, 504), (336, 504), (350, 504)]; center-cropping all to smallest (308,504)
[INFO ] Processed Images Done taking 0.17540931701660156 seconds. Shape:  torch.Size([10, 3, 308, 504])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 20.597400188446045 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0005223751068115234 seconds


[[3.9959452  4.0026364  3.9829528  ... 6.540218   6.556819   6.395972  ]
 [3.9823902  3.9181428  3.914079   ... 6.3556414  6.155      6.5334864 ]
 [3.9808552  3.915219   3.976036   ... 6.428235   6.3713593  6.496007  ]
 ...
 [0.6809618  0.68412054 0.6790412  ... 0.6705993  0.6704862  0.67812645]
 [0.68144774 0.6868372  0.6804886  ... 0.6725665  0.67334163 0.6838999 ]
 [0.6757506  0.6842188  0.68408304 ... 0.6792757  0.68014216 0.68156135]]


In [ ]:
def merge_point_clouds(prediction, confidence_threshold):
    


In [ ]:
import numpy as np
import open3d as o3d
from PIL import Image

image_path = '/home/maciej/dev/depth/422b1df6e7202cdef0cf8824c206138f.jpg'
image = Image.open(image_path)

# 1. Run model inference
prediction = model.inference(image=[image_path])

# 2. Extract arrays
depth_map = prediction.depth[0]
confidence_map = prediction.conf[0] 

# 3. Apply your threshold logic
confidence_threshold = 0.7
mask = confidence_map > confidence_threshold
clean_depth = np.where(mask, depth_map, 0.0)

# FIX: Get target size from the AI output shape (H, W)
target_height, target_width = clean_depth.shape

# Resize your color image to match the depth array exactly
# (PIL expect width, height for .resize() parameter order)
resized_image = image.resize((target_width, target_height), Image.Resampling.BILINEAR)
color_image_rgb = np.array(resized_image)

# 4. Now conversion to Open3D will pass smoothly
o3d_depth = o3d.geometry.Image(clean_depth.astype(np.float32))
o3d_color = o3d.geometry.Image(color_image_rgb)

# 5. Combine into an RGBD Image
rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
    o3d_color, 
    o3d_depth, 
    convert_rgb_to_intensity=False
)

[INFO ] Processed Images Done taking 0.016909360885620117 seconds. Shape:  torch.Size([1, 3, 420, 504])
[INFO ] Model Forward Pass Done. Time: 2.8744239807128906 seconds
[INFO ] Conversion to Prediction Done. Time: 0.00036787986755371094 seconds
Success! Math matching complete. Depth shape: (420, 504), Color shape: (420, 504, 3)


In [54]:
# Extract the estimated intrinsics for this specific frame from DA3
# (DA3 gives you a 3x3 matrix, we convert it to an Open3D Intrinsic object)
intrinsic_matrix = prediction.intrinsics[0] # Grab first frame's matrix

o3d_intrinsics = o3d.camera.PinholeCameraIntrinsic()
o3d_intrinsics.set_intrinsics(
    width=color_image_rgb.shape[1],
    height=color_image_rgb.shape[0],
    fx=intrinsic_matrix[0, 0],
    fy=intrinsic_matrix[1, 1],
    cx=intrinsic_matrix[0, 2],
    cy=intrinsic_matrix[1, 2]
)

# Build the point cloud!
pcd_frame = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, o3d_intrinsics)

# Grab the estimated camera position for this frame
extrinsic_matrix = prediction.extrinsics[0] 

# Shift the point cloud into global world coordinates
pcd_frame.transform(extrinsic_matrix)

TypeError: transform(): incompatible function arguments. The following argument types are supported:
    1. (self: open3d.cuda.pybind.geometry.Geometry3D, arg0: numpy.ndarray[numpy.float64[4, 4]]) -> open3d.cuda.pybind.geometry.Geometry3D

Invoked with: PointCloud with 211680 points., array([[ 9.9999994e-01,  2.3407285e-05, -2.7559692e-04,  1.3851056e-04],
       [-2.3442979e-05,  1.0000000e+00, -1.2951183e-04,  9.6139127e-05],
       [ 2.7559386e-04,  1.2951829e-04,  9.9999994e-01,  1.3810915e-05]],
      dtype=float32)